# EV Predictive Maintenance: Preprocessing

This notebook prepares a reproducible, time-aware classification dataset. It does not train a model or alter the raw CSV.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / 'src').exists():
    if project_root.parent == project_root:
        raise RuntimeError('Run this notebook from inside the project repository.')
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.load_data import load_dataset
from src.preprocessing.cleaning import (
    build_numeric_preprocessor,
    chronological_train_test_split,
    parse_timestamp_column,
    select_classification_features,
)
from src.preprocessing.validation import (
    dataset_quality_summary,
    ensure_binary_target,
    ensure_valid_timestamps,
)

In [ ]:
data_path = project_root / 'data' / 'raw' / 'EV_Predictive_Maintenance_Dataset_15min.csv'
raw_data = load_dataset(data_path)
data = parse_timestamp_column(raw_data)

ensure_binary_target(data, 'Failure_Probability')
ensure_valid_timestamps(data)
quality = dataset_quality_summary(data)
quality

In [ ]:
features = select_classification_features(data)
target = data['Failure_Probability'].copy()

train_data, test_data = chronological_train_test_split(data, test_size=0.20)
train_features = select_classification_features(train_data)
test_features = select_classification_features(test_data)

numeric_features = train_features.select_dtypes(include='number').columns.tolist()
preprocessor = build_numeric_preprocessor(numeric_features)

print(f'Candidate features: {len(features.columns)}')
print(f'Train rows: {len(train_data):,}; test rows: {len(test_data):,}')
print(f'Train ends: {train_data.Timestamp.max()}')
print(f'Test starts: {test_data.Timestamp.min()}')
print('Preprocessor is intentionally unfitted. Fit it only inside a future training pipeline.')

## Decisions retained for later stages

`Maintenance_Type`, `RUL`, `TTF`, and `Component_Health_Score` are excluded as potential leakage. `Timestamp` is retained only for ordering and splitting. No outlier removal, resampling, feature engineering, or model fitting occurs here. See `reports/03_Preprocessing_Report.md` for the full rationale.